# Oppitunti 11 - Mallikontekstiprotokolla (MCP)

**Mallikontekstiprotokolla (MCP)** on avoin standardi, joka mahdollistaa agenttien dynaamisen työkalujen, resurssien ja tietolähteiden löytämisen ja käyttämisen suoritusaikana. Sen sijaan, että työkalut kovakoodattaisiin agenttiin, MCP antaa agenttien yhdistyä ulkoisiin palvelimiin, jotka tarjoavat ominaisuuksia tarpeen mukaan.

Tässä oppitunnissa opit:
- Mitä MCP on ja miksi se on tärkeä agenttijärjestelmille
- Kuinka MCP-asiakas-palvelinarkkitehtuuri toimii
- Kuinka rakentaa agentteja, jotka käyttävät MCP-tyylistä työkalujen etsintää


## Asennus

**Edellytykset:**
- Microsoft Foundry -projekti, jossa on otettu käyttöön malli
- Suorita `az login` todennusta varten


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mikä on Model Context Protocol (MCP)?

MCP määrittelee vakioidun tavan tekoälyagenttien löytää ja olla vuorovaikutuksessa ulkoisten työkalujen ja tietolähteiden kanssa:

- **MCP-palvelin**: Tarjoaa työkaluja, resursseja ja kehotteita standardin protokollan kautta
- **MCP-asiakas**: Agentin ajonaikainen osa, joka yhdistää palvelimiin ja löytää saatavilla olevat ominaisuudet
- **Dynaaminen löytäminen**: Agenttien ei tarvitse käyttää kovakoodattuja työkaluja — ne löytävät käytettävissä olevat resurssit ajon aikana

Tämä on tehokas tapa rakentaa laajennettavia agenttijärjestelmiä, joissa uusia ominaisuuksia voidaan lisätä muuttamatta agenttikoodia.


## Kuinka MCP toimii

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agentti (MCP-asiakas) yhdistää MCP-palvelimeen
2. Palvelin vastaa luettelolla käytettävissä olevista työkaluista ja niiden kaavoista
3. Agentti voi sitten kutsua mitä tahansa löydettyä työkalua päättelynsä aikana
4. Tulokset kulkevat takaisin saman protokollan kautta


## MCP-työkalun löytäminen simuloimalla

Koska todellinen MCP-palvelin vaatii käynnissä olevan palveluprosessin, esittelemme kaavan käyttämällä `@tool`-funktioita, jotka simuloivat mitä MCP-yhdistetty majoituspalvelu tarjoaisi.

Tuotannossa nämä työkalut löydettäisiin dynaamisesti MCP-palvelimelta sen sijaan, että ne määritettäisiin paikallisesti.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Edustajan rakentaminen MCP-tyylisillä työkaluilla


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP tuotannossa

Tuotantoympäristössä MCP mahdollistaa tehokkaita malleja:

- **Dynaaminen työkalujen löytyminen**: Agentit yhdistyvät MCP-palvelimiin ja löytävät työkalut ajon aikana
- **Riippumaton arkkitehtuuri**: Työkalujen tarjoajat voivat päivittää itsenäisesti agentista riippumatta
- **Organisaatiorajat ylittävä jakaminen**: Tiimit voivat tarjota toiminnallisuuksia MCP-palvelimien kautta, joita kuka tahansa agentti voi käyttää
- **Microsoft Agent Frameworkin tuki**: MAF sisältää sisäänrakennetun MCP-asiakastuen `mcp`-integraation kautta

Käyttääksesi oikeaa MCP-palvelinta MAF:n kanssa, yhdistä `hosted_mcp_tool()`-funktion tai MCP-asiakasintegraation kautta.

**Lue lisää:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
